# Projeto 1: classificador de notícias

**Ciência dos Dados, 2026.2**

Integrante 1:

Integrante 2:

Integrante 3 (se trio):

---

Este notebook é o entregável principal do Projeto 1. Preencha cada etapa,
apagando as instruções em itálico e substituindo pelo seu texto e código.

Antes de entregar, releia o enunciado, confira a rubrica e limpe as saídas
exploratórias que você não discute no texto.

In [30]:
%matplotlib inline
import math
import re
import unicodedata
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

In [31]:
USERNAME = "Miachesini"

treino = pd.read_csv(f"../dados/dados_treino_{USERNAME}.csv")
teste = pd.read_csv(f"../dados/dados_teste_{USERNAME}.csv")

print(f"treino: {len(treino)} artigos | teste: {len(teste)} artigos")
print(f"categorias: {sorted(treino['categoria'].unique())}")
treino.head(3)

treino: 2425 artigos | teste: 1040 artigos
categorias: ['film', 'games', 'world']


,Documento,categoria
0,ontario's premier reversed course controversia...,world
1,vladimir putin promised free grain supplies af...,world
2,january players nintendo's legend zelda ocarin...,games


---
## Etapa 1: registro do sorteio

*Registre aqui as categorias sorteadas pelo seu grupo e a semente usada, copiadas
do notebook `Cria_base_de_dados_Treino_e_Teste_Projeto1.ipynb`. Descreva
brevemente do que trata cada categoria e comente o tamanho da base resultante:
qual foi a menor categoria sorteada e como isso limitou o balanceamento.*

ESCREVA AQUI

In [32]:
# Se quiser, imprima aqui a distribuição das categorias como evidência

---
## Etapa 2: o classificador Naive-Bayes com Laplace

*Trabalhe apenas com `treino` nesta etapa. A base de teste só entra na Etapa 4.*

### 2.1 Da limpeza às palavras

*A limpeza já foi feita no `Cria_base_de_dados_Treino_e_Teste_Projeto1.ipynb`, e é
de lá que vêm os números que você vai relatar aqui. Falta transformar cada artigo
numa lista de palavras: é a mesma `separar_palavras()` daquele notebook, repetida
abaixo para você usar.*

*Se você melhorou a limpeza (truncamento, stopwords, stemming), traga a versão
melhorada para cá e diga o que mudou.*

In [33]:
TRADUCAO = {
    0x2018: "'", 0x2019: "'", 0x201C: '"', 0x201D: '"',
    0x2013: "-", 0x2014: "-", 0x2026: " ", 0x00A0: " ",
}
PADRAO_PALAVRA = re.compile(r"[a-z]+(?:'[a-z]+)?")


def separar_palavras(texto):
    texto = texto.translate(TRADUCAO)
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return PADRAO_PALAVRA.findall(texto.lower())


palavras_treino = [separar_palavras(t) for t in treino["Documento"]]
print(f"vocabulario do treino: {len({p for d in palavras_treino for p in d}):,} palavras")
print(f"palavras por artigo (media): {sum(len(d) for d in palavras_treino) / len(palavras_treino):.0f}")

vocabulario do treino: 54,085 palavras
palavras por artigo (media): 269


*Relate aqui o efeito da limpeza: quantas linhas cada passo removeu, como o
vocabulário mudou, e o que você alterou na função entregue, com a justificativa.*

ESCREVA AQUI

### 2.2 Treinamento

*Implemente o Naive-Bayes com suavização de Laplace, equação (3) do enunciado,
para todas as categorias sorteadas.*

**Trabalhe em espaço logarítmico**, usando a equação (4). O enunciado explica por
quê: com artigos de 680 palavras, o produto das probabilidades vira exatamente
zero, o modelo passa a classificar tudo na mesma categoria e nenhum erro é
levantado.

In [34]:
palavras_por_classe = {}
for palavras, cat in zip(palavras_treino, treino["categoria"]):
    if cat not in palavras_por_classe:
        palavras_por_classe[cat] = []
    palavras_por_classe[cat].extend(palavras)

for cat, palavras in palavras_por_classe.items():
    print(f"{cat:>8}: {len(palavras):,} palavras | {len(set(palavras)):,} distintas")

   world: 225,084 palavras | 23,664 distintas
   games: 228,584 palavras | 28,715 distintas
    film: 197,764 palavras | 31,910 distintas


In [35]:
# ESCREVA AQUI o cálculo das probabilidades a priori P(c).
artigos_por_classe = {}
for cat in treino["categoria"]:
    if cat not in artigos_por_classe:
        artigos_por_classe[cat] = 0
    artigos_por_classe[cat] += 1

print(artigos_por_classe)

probabilidade_classes = {}
soma = 0
for valores in artigos_por_classe.values():
    soma += valores
for classe, valor in artigos_por_classe.items():
    probabilidade_classes[classe] = valor/soma

print(probabilidade_classes)

{'world': 808, 'games': 809, 'film': 808}
{'world': 0.3331958762886598, 'games': 0.3336082474226804, 'film': 0.3331958762886598}


In [39]:
# ESCREVA AQUI o cálculo das verossimilhanças P(w_i|c) com suavização de Laplace,
# equação (3), já em log.

V = len({p for d in palavras_treino for p in d})

contagem_por_classe = {}


# Count otimizado
for cat, lista in palavras_por_classe.items():
    contagem_por_classe[cat] = {}
    for p in lista:
        contagem_por_classe[cat][p] = contagem_por_classe[cat].get(p, 0) + 1


# Calculando P(W|C)
prob_palavras_por_classe = {}
for classes, palavrass in palavras_por_classe.items():
    prob_palavras_por_classe[classes] = {}
    for palavra in contagem_por_classe[classes]:
        
        n = contagem_por_classe[classes][palavra]
        conta = math.log((n + 1) / (len(palavrass) + V))
        prob_palavras_por_classe[classes][palavra] = conta


# Verificação de fórmula
for cat in prob_palavras_por_classe:
    N_c = len(palavras_por_classe[cat])
    soma = sum(math.exp(v) for v in prob_palavras_por_classe[cat].values())
    ausentes = V - len(prob_palavras_por_classe[cat])
    soma += ausentes * (1 / (N_c + V))
    print(f"{cat:>8}: soma = {soma:.6f}")

   world: soma = 1.000000
   games: soma = 1.000000
    film: soma = 1.000000


In [44]:
# ESCREVA AQUI a função que classifica um artigo pelo maior escore da equação (4).

# Conjunto de todas as palavras vistas no treino. Serve para distinguir
# "palavra desconhecida" de "palavra conhecida, mas ausente nesta categoria".
vocabulario = {p for d in palavras_treino for p in d}

# N_c da equação (3): total de palavras de cada categoria, com repetição.
# Pré-calculado porque é consultado dentro de dois loops aninhados.
N_por_classe = {cat: len(lista) for cat, lista in palavras_por_classe.items()}


def classificacao(palavras):
    """Recebe a lista de palavras de um artigo e devolve a categoria de maior escore."""
    escores = {}

    for cat in prob_palavras_por_classe:
        N_c = N_por_classe[cat]

        # Primeiro termo da equação (4): log P(c), a probabilidade a priori.
        # Inicializar aqui também cria a chave, evitando KeyError no += abaixo.
        escores[cat] = math.log(probabilidade_classes[cat])

        for word in palavras:

            # Caso 1: a palavra ocorre nesta categoria. O valor guardado já está
            # em log (equação 3 aplicada no treino), então soma direto.
            if word in prob_palavras_por_classe[cat]:
                escores[cat] += prob_palavras_por_classe[cat][word]

            # Caso 2: a palavra existe no treino, mas nunca nesta categoria.
            # A contagem é zero e o Laplace dá (0+1)/(N_c+V). É justamente o
            # caso que a suavização existe para tratar: a ausência é evidência
            # contra a categoria e precisa pesar no escore.
            elif word in vocabulario:
                escores[cat] += math.log(1 / (N_c + V))

            # Caso 3: palavra nunca vista no treino. Não está no vocabulário,
            # logo não é evento do modelo e é ignorada. Como penalizaria todas
            # as categorias de forma equivalente, não alteraria a decisão.

    # Equação (4) decide pelo maior escore. O key=escores.get faz o max comparar
    # pelos valores, mas devolver a chave (o nome da categoria).
    return max(escores, key=escores.get)

---
## Etapa 3: a variante com TF-IDF

*Construa a segunda versão do classificador, ponderando a contagem de cada palavra
pelo peso TF-IDF, conforme as equações (5), (6) e (7) do enunciado.*

*Lembre: M e m_i são contados apenas na base de treino.*

In [46]:
# ESCREVA AQUI o cálculo do IDF de cada palavra, a partir do treino.
M = len(palavras_treino)

# m_i: em quantos artigos cada palavra aparece. O set(d) é essencial:
# conta presença no artigo, não número de ocorrências.
docs_por_palavra = {}
for d in palavras_treino:
    for p in set(d):
        docs_por_palavra[p] = docs_por_palavra.get(p, 0) + 1

idf = {p: math.log(M / (1 + m)) for p, m in docs_por_palavra.items()}

In [ ]:
# ESCREVA AQUI a versão do classificador com as contagens ponderadas, equações (6) e (7).

# Equação (6): n~_i|c = n_i|c * IDF(w_i)
contagem_tfidf = {}
for cat, contagens in contagem_por_classe.items():
    contagem_tfidf[cat] = {}
    for p, n in contagens.items():
        contagem_tfidf[cat][p] = n * idf[p]

# N~_c da equação (7): soma de todas as contagens ponderadas da categoria
N_tfidf = {}
for cat in contagem_tfidf:
    soma = 0
    for valor in contagem_tfidf[cat].values():
        soma += valor
    N_tfidf[cat] = soma

# Equação (7): Laplace igual ao da equação (3), com as contagens ponderadas
log_prob_tfidf = {}
for cat, contagens in contagem_tfidf.items():
    log_prob_tfidf[cat] = {}
    for palavra, n_til in contagens.items():
        log_prob_tfidf[cat][palavra] = math.log((n_til + 1) / (N_tfidf[cat] + V))


def classificacao_tfidf(palavras):
    """Mesma equação (4) da Etapa 2, com as verossimilhanças ponderadas por TF-IDF."""
    escores = {}
    for cat in log_prob_tfidf:
        escores[cat] = math.log(probabilidade_classes[cat])
        for word in palavras:
            if word in log_prob_tfidf[cat]:
                escores[cat] += log_prob_tfidf[cat][word]
            elif word in vocabulario:
                escores[cat] += math.log(1 / (N_tfidf[cat] + V))
    return max(escores, key=escores.get)

In [51]:
# ESCREVA AQUI as palavras de menor e de maior IDF.
ordenado = sorted(idf.items(), key=lambda x: x[1])

print("menor IDF (aparecem em quase todo artigo):")
for p, v in ordenado[:15]:
    print(f"  {p:>15}: IDF = {v:.4f}  ({docs_por_palavra[p]} artigos)")

print("\nmaior IDF (raras):")
for p, v in ordenado[-15:]:
    print(f"  {p:>15}: IDF = {v:.4f}  ({docs_por_palavra[p]} artigos)")

menor IDF (aparecem em quase todo artigo):
              new: IDF = 0.6969  (1207 artigos)
             it's: IDF = 0.7018  (1201 artigos)
             time: IDF = 0.7094  (1192 artigos)
             like: IDF = 0.7195  (1180 artigos)
             said: IDF = 0.7743  (1117 artigos)
           people: IDF = 0.8200  (1067 artigos)
            image: IDF = 0.8295  (1057 artigos)
             view: IDF = 0.8739  (1011 artigos)
       photograph: IDF = 0.9215  (964 artigos)
             just: IDF = 0.9246  (961 artigos)
             year: IDF = 0.9435  (943 artigos)
            world: IDF = 0.9584  (929 artigos)
            years: IDF = 1.0401  (856 artigos)
             game: IDF = 1.0794  (823 artigos)
              way: IDF = 1.1216  (789 artigos)

maior IDF (raras):
      queensferry: IDF = 7.1004  (1 artigos)
          situate: IDF = 7.1004  (1 artigos)
          sykes's: IDF = 7.1004  (1 artigos)
        servicing: IDF = 7.1004  (1 artigos)
        trackside: IDF = 7.1004  (1 artigos)

*A lista de palavras com menor e maior IDF faz sentido, dado o assunto das
categorias do seu grupo? Comente.*

ESCREVA AQUI

---
## Etapa 4: análise de performance

*Agora sim, use `teste`. Avalie as duas versões do classificador, a da Etapa 2 e a
da Etapa 3.*

In [55]:
# ESCREVA AQUI a matriz de confusão N x N das duas versões.
palavras_teste = [separar_palavras(t) for t in teste["Documento"]]
y_verdadeiro = list(teste["categoria"])

y_nb = []
for p in palavras_teste:
    y_nb.append(classificacao(p))

y_tfidf = []
for p in palavras_teste:
    y_tfidf.append(classificacao_tfidf(p))

categorias = sorted(probabilidade_classes.keys())


# linha = categoria real, coluna = categoria prevista. A diagonal são os acertos.
def matriz_confusao(verdadeiros, previstos):
    m = {}
    for v in categorias:
        m[v] = {}
        for p in categorias:
            m[v][p] = 0

    for v, p in zip(verdadeiros, previstos):
        m[v][p] += 1

    return m


def imprimir_matriz(m, titulo):
    print(titulo)
    print("real \\ previsto", categorias)
    for real in categorias:
        linha = []
        for prev in categorias:
            linha.append(m[real][prev])
        print(f"{real:>8}", linha)
    print()


mc_nb = matriz_confusao(y_verdadeiro, y_nb)
mc_tfidf = matriz_confusao(y_verdadeiro, y_tfidf)

imprimir_matriz(mc_nb, "Naive-Bayes (Etapa 2)")
imprimir_matriz(mc_tfidf, "Naive-Bayes + TF-IDF (Etapa 3)")

Naive-Bayes (Etapa 2)
real \ previsto ['film', 'games', 'world']
    film [336, 4, 7]
   games [2, 343, 1]
   world [17, 2, 328]

Naive-Bayes + TF-IDF (Etapa 3)
real \ previsto ['film', 'games', 'world']
    film [330, 8, 9]
   games [2, 343, 1]
   world [15, 3, 329]



In [56]:
# ESCREVA AQUI as porcentagens de VP, FP, VN e FN por categoria
# (abordagem "um contra os demais"), para as duas versões.
def metricas(m, titulo):
    print(titulo)

    # total de artigos da base de teste
    total = 0
    for v in categorias:
        for p in categorias:
            total += m[v][p]

    for c in categorias:
        # VP: era c e o modelo disse c (a diagonal)
        vp = m[c][c]

        # FN: era c, mas o modelo mandou para outra categoria (resto da linha)
        fn = 0
        for p in categorias:
            if p != c:
                fn += m[c][p]

        # FP: não era c, mas o modelo disse c (resto da coluna)
        fp = 0
        for v in categorias:
            if v != c:
                fp += m[v][c]

        # VN: todo o resto
        vn = total - vp - fn - fp

        print(f"{c:>8}: VP={100*vp/total:.1f}%  FP={100*fp/total:.1f}%  VN={100*vn/total:.1f}%  FN={100*fn/total:.1f}%")

    print()


metricas(mc_nb, "Naive-Bayes (Etapa 2)")
metricas(mc_tfidf, "Naive-Bayes + TF-IDF (Etapa 3)")

Naive-Bayes (Etapa 2)
    film: VP=32.3%  FP=1.8%  VN=64.8%  FN=1.1%
   games: VP=33.0%  FP=0.6%  VN=66.2%  FN=0.3%
   world: VP=31.5%  FP=0.8%  VN=65.9%  FN=1.8%

Naive-Bayes + TF-IDF (Etapa 3)
    film: VP=31.7%  FP=1.6%  VN=65.0%  FN=1.6%
   games: VP=33.0%  FP=1.1%  VN=65.7%  FN=0.3%
   world: VP=31.6%  FP=1.0%  VN=65.7%  FN=1.7%



In [57]:
# ESCREVA AQUI a acurácia geral das duas versões.
def acuracia(verdadeiros, previstos):
    acertos = 0
    for v, p in zip(verdadeiros, previstos):
        if v == p:
            acertos += 1
    return acertos / len(verdadeiros)


acc_nb = acuracia(y_verdadeiro, y_nb)
acc_tfidf = acuracia(y_verdadeiro, y_tfidf)

# baseline: acurácia de um modelo que chutasse sempre a categoria mais frequente
baseline = max(probabilidade_classes.values())

print(f"acurácia Naive-Bayes:          {acc_nb:.4f}")
print(f"acurácia Naive-Bayes + TF-IDF: {acc_tfidf:.4f}")
print(f"diferença: {acc_tfidf - acc_nb:+.4f}")
print(f"baseline (chute na maior classe): {baseline:.4f}")

acurácia Naive-Bayes:          0.9683
acurácia Naive-Bayes + TF-IDF: 0.9635
diferença: -0.0048
baseline (chute na maior classe): 0.3336


### 4.1 Interpretação

*Qual categoria o modelo acerta melhor, e qual ele erra mais? Por quê? Qual par de
categorias se confunde mais, e a confusão é simétrica ou tem direção?*

*A ponderação TF-IDF melhorou o resultado? Em quais categorias? Compare as duas
matrizes de confusão, não só as duas acurácias. Se o TF-IDF piorou, isso é um
achado válido: investigue e explique.*

ESCREVA AQUI

---
## Etapa 5: validação cruzada e estabilidade

*Junte treino e teste, aplique Stratified K-Fold com k = 10 e recalcule a acurácia
em cada divisão. Aqui basta o classificador da Etapa 2.*

In [ ]:
# ESCREVA AQUI a validação cruzada estratificada com k = 10.

In [ ]:
# ESCREVA AQUI as medidas-resumo das 10 acurácias.

*Quão estável é o seu classificador? A acurácia da Etapa 4 caiu dentro do
intervalo observado nas 10 divisões? E a diferença entre o modelo com e sem
TF-IDF: ela é maior ou menor do que a variação entre divisões?*

ESCREVA AQUI

---
## Referências

*Liste aqui as referências que você efetivamente usou.*